# Estadísticas por municipio

In [0]:
import pyspark.sql.functions as F

In [0]:
# Educacion (estadisticas) por municipio https://www.datos.gov.co/Educaci-n/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR-B-SICA/nudc-7mev/about_data
mpio = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/workspace/pdge(saber-11)/saber11/BRONZE/mpio.csv")

In [0]:
mpio.printSchema()

In [0]:
# Conteo de nulos
mpio.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in mpio.columns]).show()

In [0]:
# Inconsistencias case-sensitive
for c in mpio.columns:
    mpio.select(F.count(F.when(F.col(c) != F.lower(F.col(c)),True))).show()

In [0]:
# Conteo de duplicados
total     = mpio.count()
distintos = mpio.distinct().count()

print(f"Total filas:     {total}")
print(f"Filas distintas: {distintos}")
print(f"Duplicados:      {total - distintos}")

In [0]:
#for c in mpio.columns:
    #mpio.groupBy(c).count().show()

## Filtrado

In [0]:
nuevo = mpio

In [0]:
nuevo = nuevo.filter(F.col("AÑO").between(2015, 2023))

In [0]:
cols = ["SEDES_CONECTADAS_A_INTERNET","CÓDIGO_ETC","ETC","TAMAÑO_PROMEDIO_DE_GRUPO"]
for c in cols:
    nuevo = nuevo.drop(c)

In [0]:
nuevo.dropna()

In [0]:
nuevo.count()

## Transformaciones

El dataset cuenta con valores agregados de porcentajes de deserión, aprovación, reprovación y repitiencia agrupados por nivel educativo.

In [0]:
nuevo = nuevo.withColumnRenamed("CÓDIGO_MUNICIPIO","COD_MUNICIPIO" )

In [0]:
nuevo.write.mode("overwrite").parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/mpio/")